# LLM Pretraining — the end-to-end process

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adiel2012/llm-pretraining/blob/main/llm-pretraining.ipynb)

This notebook is the runnable version of **Part 16** of `README.md`: raw
text → filtered and deduplicated corpus → BPE tokenizer → binary token files → a modern
decoder-only transformer → a fully instrumented training run → checkpointing →
evaluation → a controlled annealing experiment → a mini scaling law.

Every stage has the same shape: **Goal**, **What's happening**, **Predict before
you run** (write your guess down first), the **code**, then **Verify** (a check
you must pass before moving on) and **Breaks like this** (the failure signatures
to recognize). Parts 1–15 of the guide explain the *why* behind each stage.

### Before you start

1. **Runtime → Change runtime type → T4 GPU** (or better). Everything runs on CPU
   at the `tiny` preset, but slowly.
2. The notebook ships on the **`tiny` preset**: a smoke test that runs top to
   bottom in minutes and proves the pipeline works. Its numbers are *not* meant
   to be good. For the real run, change `PRESET = "small"` in Stage 1 and set
   `USE_DRIVE = True` in the setup cell.

| Preset | Model | Tokens | Time on a T4 | Purpose |
|---|---|---|---|---|
| `tiny` | ~0.8M non-emb params | ~4M | minutes | prove the pipeline works |
| `small` | ~22M | ~295M | 3–5 h for Stage 11, plus Stages 14–15 | the real run |
| `base` | ~75M | ~1.5B | A100-class, most of a day | wants FineWeb, not TinyStories |

3. **Cell order.** Run top to bottom. The one ordering quirk from the guide is
   already handled: the checkpoint helpers (Stage 12a) sit *above* Stage 10,
   because the training loop calls them.
4. **Disconnects.** Free Colab sessions drop. With `USE_DRIVE = True`, data and
   checkpoints live on Google Drive, and the **Appendix** at the bottom resumes
   training without redoing Stages 2–6.

## Stage 0 — Colab setup

Check the GPU, install the two packages Colab doesn't always have, and optionally
put all data and checkpoints on Google Drive so they survive a disconnect.

In [ ]:
# Cell 0 — Colab setup
import sys, os
IN_COLAB = "google.colab" in sys.modules

!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU visible"
# torch, numpy, scipy, tqdm and matplotlib are preinstalled on Colab.
!pip install -q datasets tokenizers ftfy

# True = keep data + checkpoints on Google Drive (survives disconnects; strongly
# recommended for `small` and `base`). False = fast local disk, lost on disconnect.
USE_DRIVE = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/llm-pretraining"
elif IN_COLAB:
    ROOT = "/content/llm-pretraining"
else:
    ROOT = "llm-pretraining"
os.makedirs(ROOT, exist_ok=True)
print(f"working directory for data and runs: {ROOT}")

## Stage 1 — Setup and config

**Goal.** One config object every later cell reads from, with size presets so the
same notebook runs on a laptop or an A100.

**What's happening.** Pretraining has ~30 coupled hyperparameters. Fixing them in
one place — and deriving everything you can rather than hardcoding it — is the
difference between a notebook you can experiment with and one you can only run
once. The presets let you debug at `tiny` and then scale the exact same code.

**Predict before you run.** Write down your guesses, then check the printout:

1. How many tokens per optimizer step? (`batch_size × grad_accum × seq_len`)
2. Roughly how many non-embedding parameters? Full multi-head attention is
   ~`4·d²` per layer, but GQA shrinks K and V by `n_kv_head/n_head`: with 2 of 8
   heads that is `2.5·d²`. The SwiGLU MLP is ~`8·d²`. So ~`10.5·d²·n_layer`.
3. Divide total tokens by that. Are you above or below Chinchilla's ~20?

In [ ]:
# Cell 1 — config
import math, os, time, json, random
from dataclasses import dataclass, asdict, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F

@dataclass
class Config:
    # --- data ---
    dataset: str = "roneneldan/TinyStories"   # swap for a FineWeb slice later
    n_docs: int = 400_000
    data_dir: str = "data"

    # --- tokenizer ---
    vocab_size: int = 16_000

    # --- model ---
    d_model: int = 512
    n_layer: int = 8
    n_head: int = 8
    n_kv_head: int = 2          # GQA: n_head must be divisible by this
    seq_len: int = 512
    rope_theta: float = 10_000.0
    ffn_multiple: int = 128     # round SwiGLU hidden dim to this
    tie_embeddings: bool = True

    # --- optimization ---
    batch_size: int = 24        # micro-batch (per forward)
    grad_accum: int = 4         # tokens/step = batch_size*grad_accum*seq_len
    lr: float = 6e-4
    min_lr_frac: float = 0.1
    warmup_frac: float = 0.02
    decay_frac: float = 0.20    # WSD: final fraction spent decaying
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    max_steps: int = 6_000

    # --- runtime ---
    seed: int = 1337
    compile: bool = True
    eval_every: int = 250
    ckpt_every: int = 1_000
    out_dir: str = "runs/small"

PRESETS = {
    # ~0.8M non-emb params — CPU-viable. A SMOKE TEST, not a real run:
    # deliberately far below Chinchilla, just enough to prove the pipeline works.
    "tiny":  dict(d_model=128, n_layer=4,  n_head=4,  n_kv_head=2,
                  seq_len=256, vocab_size=8_000, batch_size=8, grad_accum=1,
                  max_steps=2_000, lr=1e-3, n_docs=20_000, out_dir="runs/tiny"),
    # ~22M non-emb params, ~295M tokens (~13 tok/param).
    # ~1h on an A100, more like 3-5h on a T4.
    "small": dict(),
    # ~75M non-emb params, ~1.5B tokens (~20 tok/param = Chinchilla).
    # A100-class, the better part of a day. Really wants FineWeb, not TinyStories.
    "base":  dict(d_model=768, n_layer=12, n_head=12, n_kv_head=4,
                  seq_len=1024, vocab_size=32_000, batch_size=24, grad_accum=8,
                  max_steps=7_700, lr=4e-4, n_docs=2_000_000, out_dir="runs/base"),
}

PRESET = "tiny"          # <-- the one knob: "tiny" (smoke test) | "small" | "base"
cfg = Config(**PRESETS[PRESET])
# Colab: put everything under ROOT (Stage 0). Data is per-preset because the
# tokenizer's vocab size differs between presets.
cfg.data_dir = f"{ROOT}/data/{PRESET}"
cfg.out_dir  = f"{ROOT}/{cfg.out_dir}"

torch.manual_seed(cfg.seed); np.random.seed(cfg.seed); random.seed(cfg.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(cfg.data_dir, exist_ok=True); os.makedirs(cfg.out_dir, exist_ok=True)

# bf16 needs Ampere (sm_80+). A T4 is Turing and must use fp16 + GradScaler.
# Do NOT use torch.cuda.is_bf16_supported(): by default it counts *emulated*
# bf16 and returns True on a T4, which then runs bf16 slowly in software.
use_bf16   = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8
amp_dtype  = torch.bfloat16 if use_bf16 else torch.float16
gpu_name   = torch.cuda.get_device_name(0) if device == "cuda" else "cpu"

TOKENS_PER_STEP = cfg.batch_size * cfg.grad_accum * cfg.seq_len
total_tokens    = TOKENS_PER_STEP * cfg.max_steps
print(json.dumps(asdict(cfg), indent=2))
print(f"\ndevice={device} ({gpu_name})  amp={amp_dtype}")
print(f"tokens/step={TOKENS_PER_STEP:,}   total tokens={total_tokens/1e6:.0f}M")

**Verify.** `n_head % n_kv_head == 0`, and `device` is `cuda` (a silent CPU
fallback is the most common Colab mistake). Then compare your three guesses: for
`small` you should land near 49k tokens/step, ~22M non-embedding parameters and
~13 tokens/parameter — deliberately below Chinchilla. `tiny` is far below it on
purpose.

**Breaks like this.** Silent CPU fallback. Assuming bf16 exists — on a T4 it does
not. `seq_len` set beyond what memory allows, which won't fail until Stage 11.

In [ ]:
assert cfg.n_head % cfg.n_kv_head == 0, "GQA needs n_head divisible by n_kv_head"
assert cfg.d_model % cfg.n_head == 0, "d_model must divide evenly into heads"
if device != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU. "
          "Only the `tiny` preset is practical on CPU.")

## Stage 2 — Acquire raw text

**Goal.** A list of raw document strings.

**What's happening.** Every pretraining corpus starts as documents, not tokens.
TinyStories is the right starting corpus because it is small, clean, and a
30M-param model can genuinely master it — you will see coherent English, which is
the motivating payoff. Swap in a FineWeb slice once the pipeline works end to end.

In [ ]:
# Cell 2 — raw text
from datasets import load_dataset

# TinyStories: clean, tiny, learnable at 30M params.
ds = load_dataset(cfg.dataset, split="train", streaming=True)
docs = []
for i, ex in enumerate(ds):
    if i >= cfg.n_docs: break
    docs.append(ex["text"])

# --- swap for real web data once the pipeline runs: ---
# ds = load_dataset("HuggingFaceFW/fineweb", name="sample-10BT",
#                   split="train", streaming=True)
# docs = [ex["text"] for i, ex in zip(range(cfg.n_docs), ds)]

print(f"{len(docs):,} docs, {sum(len(d) for d in docs)/1e6:.1f}M chars")

# The TinyStories copy on the Hub contains MOJIBAKE: UTF-8 that was decoded as
# cp1252 somewhere upstream, so “ arrives as "â€œ". Three documents read in
# Stage 2 looked clean; Stage 3's "read what you threw away" printout is what
# exposed it (the rejected documents were full of it). Repair it here, before
# anything filters or tokenizes -- otherwise the tokenizer spends merges on it.
# ftfy.fix_text also uncurls quotes, which is what Stage 3's punctuation rule expects.
import ftfy
n_fixed = 0
for i, d in enumerate(docs):
    fixed = ftfy.fix_text(d)
    n_fixed += (fixed != d)
    docs[i] = fixed
print(f"repaired text in {n_fixed:,} / {len(docs):,} docs")

**Verify.** Read three full documents. You are looking for encoding mojibake, HTML
remnants and truncation. Ten seconds here saves a retrain -- but three documents
is a small sample, and on TinyStories it missed the mojibake entirely: only the
rejected-document printout in Stage 3 revealed it. The cell below also counts
remaining `â€` sequences, which should be 0 after the repair.

**Breaks like this.** Streaming datasets silently yielding fewer docs than asked;
a field name that isn't `"text"` for your chosen dataset.

In [ ]:
assert len(docs) == cfg.n_docs, f"asked for {cfg.n_docs:,}, got {len(docs):,}"
print("docs still containing mojibake ('â€'):", sum("â€" in d for d in docs))
for d in docs[:3]:
    print(d, "\n" + "-" * 60)

## Stage 3 — Inspect and filter

**Goal.** A filtered document list, plus a record of what each rule removed.

**What's happening.** Raw web text is mostly junk: navigation bars, keyword spam,
truncated boilerplate. Heuristic filters are cheap and interpretable, so they run
first. The important discipline is measuring what each rule *removes*, because
over-filtering quietly destroys diversity (guide, Part 3.2).

**Predict before you run.** TinyStories is already clean. What keep rate do you
expect, and which rule will fire most?

In [ ]:
# Cell 3 — heuristic quality filters (a small Gopher/C4-style subset)
import re
from collections import Counter

STOPWORDS = {"the","be","to","of","and","that","have","with","this","it","is","in"}
TERMINAL  = (".", "!", "?", '"', "'", "”", "’")   # curly closers too, in case text isn't uncurled

def quality_signals(doc):
    words = doc.split()
    n = len(words)
    lines = [l for l in doc.split("\n") if l.strip()]
    return {
        "n_words":        n,
        "mean_word_len":  (sum(map(len, words)) / n) if n else 0.0,
        "symbol_ratio":   (doc.count("#") + doc.count("...")) / max(n, 1),
        "frac_terminal":  (sum(l.rstrip().endswith(TERMINAL) for l in lines)
                           / max(len(lines), 1)),
        "n_stopwords":    sum(w.lower() in STOPWORDS for w in words),
        "frac_dup_lines": 1 - (len(set(lines)) / max(len(lines), 1)),
    }

RULES = {   # name -> predicate that is True when the doc should be DROPPED
    # Deliberately looser than Gopher's ~50-word minimum (guide, Part 3.2):
    # TinyStories are short by design. On web text, use ~50.
    "too_short":      lambda s: s["n_words"] < 30,
    "too_long":       lambda s: s["n_words"] > 100_000,
    "odd_word_len":   lambda s: not (3.0 <= s["mean_word_len"] <= 10.0),
    "symbol_spam":    lambda s: s["symbol_ratio"] > 0.10,
    "no_punctuation": lambda s: s["frac_terminal"] < 0.60,
    "few_stopwords":  lambda s: s["n_stopwords"] < 2,
    "repetitive":     lambda s: s["frac_dup_lines"] > 0.30,
}

kept, removed, counts = [], [], Counter()
for d in docs:
    s = quality_signals(d)
    fired = [name for name, rule in RULES.items() if rule(s)]
    if fired:
        counts.update(fired); removed.append((fired, d))   # keep EVERY rule that fired
    else:
        kept.append(d)

print(f"kept {len(kept):,} / {len(docs):,}  ({100*len(kept)/len(docs):.1f}%)")
for name, c in counts.most_common():
    print(f"  {name:<16} {c:>7,}  ({100*c/len(docs):.2f}%)")

# THE IMPORTANT PART: read what you threw away.
for fired, d in random.sample(removed, min(5, len(removed))):
    print(f"\n--- dropped by {', '.join(fired)} ---\n{d[:300]}")

**Verify.** Keep rate should be 50–90% on clean data, 10–40% on raw web. Read the
five rejected documents above. If any look like text you'd want the model to
learn, loosen that rule.

**Breaks like this.** A single over-aggressive rule eating most of the corpus —
which the per-rule counter makes obvious, and an aggregate keep-rate would hide.

## Stage 4 — Deduplicate

**Goal.** Near-duplicate documents removed, and a shuffled train/validation
split fixed before anything downstream -- including the tokenizer -- sees it.

**What's happening.** Duplicated text causes memorization and wastes compute.
MinHash estimates Jaccard similarity between documents cheaply: shingle each doc
into n-grams, hash them, keep the minimum hash per permutation, then band the
signature so similar docs collide in a bucket (guide, Part 3.3). The split
happens here, not in Stage 6, because Stage 5 trains a tokenizer next -- a
tokenizer trained on documents that end up in validation has a mild but real
form of leakage.

**Predict before you run.** With `BANDS=16` and `ROWS=8`, where is the
catch-probability curve steepest? That point is `(1/BANDS)^(1/ROWS)` — compute it
before looking. Then use `1 − (1 − sʳ)ᵇ` (guide, Part 3.3) to find how often a
pair *at* that similarity gets caught; it is not 50%.

In [ ]:
# Cell 4 — MinHash LSH near-duplicate removal
import hashlib

NUM_PERM, N_GRAM, BANDS = 128, 5, 16
ROWS      = NUM_PERM // BANDS
THRESHOLD = (1 / BANDS) ** (1 / ROWS)      # steepest point of the S-curve: ~0.71
JACCARD_MIN = 0.7      # cutoff on the ESTIMATED Jaccard (fraction of matching
                       # signature entries) -- not exact; with 128 permutations
                       # the standard error is ~0.04 at s=0.7. (LSH only proposes
                       # candidates; on a corpus this small you could confirm each
                       # candidate with the exact shingle-set Jaccard instead.)

rng = np.random.default_rng(cfg.seed)
# Work in the field Z_p with p = 2**31 - 1 (a Mersenne prime). Then a, b, h are
# all < p < 2**31, so a*h + b < 2**62 fits in uint64: no overflow, and
# h -> (a*h + b) mod p is a genuine random permutation of Z_p for a != 0, drawn
# uniformly over the WHOLE field. (Restricting a to a small range while keeping
# a much larger modulus is NOT equivalent -- see "Breaks like this".)
MERSENNE = np.uint64((1 << 31) - 1)
A = rng.integers(1, int(MERSENNE), NUM_PERM, dtype=np.uint64)   # a in [1, p-1]
B = rng.integers(0, int(MERSENNE), NUM_PERM, dtype=np.uint64)   # b in [0, p-1]

def signature(doc):
    toks = doc.lower().split()
    if len(toks) < N_GRAM:
        shingles = {" ".join(toks)}
    else:
        shingles = {" ".join(toks[i:i+N_GRAM]) for i in range(len(toks)-N_GRAM+1)}
    # blake2b, not hash(): Python's hash() is salted per process, so a notebook
    # restart would silently give different results.
    # 32-bit digest reduced into Z_p: two distinct shingles collide with
    # probability ~2**-31, negligible for documents of a few hundred shingles.
    h = np.array([int.from_bytes(hashlib.blake2b(s.encode(), digest_size=4).digest(),
                                 "big") % int(MERSENNE) for s in shingles], dtype=np.uint64)
    return ((A[:, None] * h[None, :] + B[:, None]) % MERSENNE).min(axis=1)

from tqdm.auto import tqdm
sigs = [signature(d) for d in tqdm(kept, desc="minhash")]

buckets, dup_of = {}, {}       # bucket key -> LIST of kept doc ids
for i, sig in enumerate(sigs):
    keys = [(b, sig[b*ROWS:(b+1)*ROWS].tobytes()) for b in range(BANDS)]
    for key in keys:
        # A band collision is a CANDIDATE, not a duplicate. Verify each
        # candidate with the signature Jaccard estimate. A bucket holds a
        # LIST: keeping only its first member would compare every later
        # document against that one alone, and a true near-duplicate of the
        # second member would slip through.
        for cand in buckets.get(key, ()):
            est = float((sigs[i] == sigs[cand]).mean())
            if est >= JACCARD_MIN:
                dup_of[i] = (cand, est); break
        if i in dup_of: break
    if i not in dup_of:                       # only survivors join the buckets
        for key in keys:
            buckets.setdefault(key, []).append(i)

deduped = [d for i, d in enumerate(kept) if i not in dup_of]
print(f"threshold≈{THRESHOLD:.2f} (candidates kept as duplicates at estimated Jaccard ≥{JACCARD_MIN})")
print(f"removed {len(dup_of):,} near-duplicates -> {len(deduped):,} docs")

if dup_of:                       # always eyeball a matched pair
    i, (j, est) = next(iter(dup_of.items()))
    print(f"\nestimated Jaccard {est:.3f}")
    print(f"DUP:\n{kept[i][:200]}\n\nORIGINAL:\n{kept[j][:200]}")

# Shuffle before splitting: crawl/streaming order (source, time, ...) would
# otherwise leak into which documents land in train vs. val if we just sliced
# the unshuffled list. `random`, not np.random, to match the seed type Stage 1
# already uses everywhere else. (A seeded shuffle is enough for a corpus fixed
# once; large evolving corpora usually assign the split by a content hash so
# documents keep their split as new data arrives.)
rng_split = random.Random(cfg.seed)
shuffled = deduped[:]
rng_split.shuffle(shuffled)
split = int(0.995 * len(shuffled))
# Keep these named: EVERY later stage -- the tokenizer (Stage 5), the shard
# writer (Stage 6), the anneal corpus (Stage 14) -- must draw from train_docs,
# never val_docs.
train_docs, val_docs = shuffled[:split], shuffled[split:]
print(f"\ntrain/val split: {len(train_docs):,} / {len(val_docs):,} docs")

**Verify.** The printed pair really is a near-duplicate. On TinyStories expect a
few percent removed; on raw Common Crawl, 30–60% is normal.

**Breaks like this.** Treating a band collision as a confirmed duplicate — LSH
gives you *candidates*, and skipping verification deletes unrelated documents.
The mirror-image bug: one document per bucket, so a candidate that fails
verification shadows every later document in that bucket (buckets are lists).
A degenerate hash family: keeping `a, b < 2^31` but reducing modulo a much larger
prime made all 128 "permutations" nearly the same ordering -- the estimate stayed
unbiased but its standard deviation was ~0.27 instead of ~0.04 (measured at true
Jaccard 0.7). Match the field and the coefficient range (`p = 2^31 - 1`); check an
estimator's *variance*, not just its mean. Using Python's `hash()`, which changes
between sessions. Splitting after
shuffling but training the tokenizer on the unsplit corpus -- the bug this
cell's ordering exists to avoid.

## Stage 5 — Train the tokenizer

**Goal.** A BPE tokenizer saved to disk.

**What's happening.** Byte-level BPE starts from bytes and repeatedly merges the most
frequent adjacent pair until it hits the vocab size. Train it on *your* corpus —
a mismatched tokenizer taxes every token you will ever process (guide, Part 4).
Train it on `train_docs` specifically, not the full `deduped` set -- Stage 4
already split them for exactly this reason.

**Predict before you run.** What fertility (tokens per word) do you expect on
simple English? How will `1987` be split?

In [ ]:
# Cell 5 — BPE tokenizer
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

tok = Tokenizer(models.BPE(unk_token=None))
tok.pre_tokenizer = pre_tokenizers.Sequence([
    # Split digits out individually: measurably improves arithmetic [Part 4].
    pre_tokenizers.Digits(individual_digits=True),
    # ByteLevel applies the GPT-2 word/punctuation regex and maps bytes -> chars,
    # which guarantees no input is ever out-of-vocabulary.
    pre_tokenizers.ByteLevel(add_prefix_space=False),
])
tok.decoder = decoders.ByteLevel()

SPECIALS = ["<|endoftext|>"] + [f"<|reserved_{i}|>" for i in range(8)]
trainer = trainers.BpeTrainer(vocab_size=cfg.vocab_size,
                              special_tokens=SPECIALS,
                              initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                              show_progress=True)
tok.train_from_iterator(train_docs, trainer=trainer, length=len(train_docs))
tok.save(f"{cfg.data_dir}/tokenizer.json")

EOT = tok.token_to_id("<|endoftext|>")
# Stage 6 picks uint16 vs uint32 from this number automatically; this assert
# just fails loudly and immediately if a vocab_size change breaks that.
assert tok.get_vocab_size() <= 2**32 - 1, "vocab size does not fit in uint32"

# fertility = tokens per word: the number that decides your effective compute
sample = val_docs[:2000]       # held-out text, as Part 4 advises for comparing tokenizers
n_tok = sum(len(tok.encode(d).ids) for d in sample)
n_word = sum(len(d.split()) for d in sample)
print(f"vocab={tok.get_vocab_size()}  fertility={n_tok/n_word:.3f} tok/word")
print(tok.encode("The year 1987 cost $4.50.").tokens)

**Verify.** Encoding then decoding must give back the exact input, including
punctuation and unicode. Fertility should be ~1.2–1.5 for English, and digits
should appear one per token in the printed list.

**Breaks like this.** Forgetting to reserve special-token slots (adding them later
means resizing embeddings); training the tokenizer on unfiltered data so merges
are spent on junk; training it on `deduped` instead of `train_docs`, which
quietly reintroduces the leakage Stage 4 split to avoid.

In [ ]:
for s in ["The year 1987 cost $4.50.",
          "Ünïcödé — “curly quotes”, tabs\tand emoji 🙂",
          train_docs[0][:500]]:
    assert tok.decode(tok.encode(s).ids) == s, f"round-trip failed: {s!r}"
print("round-trip OK")

## Stage 6 — Tokenize to binary

**Goal.** `train.bin` and `val.bin` — flat token-id arrays, written incrementally.

**What's happening.** Training reads tokens millions of times, so you tokenize
once, up front, into a flat binary file that can be memory-mapped. Documents are
concatenated with an end-of-text token between them (guide, Part 3.6).
`train_docs`/`val_docs` already exist from Stage 4 -- this stage only encodes
them, it does not decide the split. This produces two monolithic files, not
the sharded layout a real pipeline would use at scale -- one file per split is
enough at TinyStories size. `CHUNK` only controls how many documents go into
one `tok.encode_batch()` call; nothing is held in an array, so it does not
bound memory.

**Predict before you run.** How many equivalent token epochs will this run
make (batches are sampled with replacement, so these are not literal
sequential passes)? More than ~4 gives diminishing returns (guide, Part 3.5).

In [ ]:
# Cell 6 — tokenize to flat binary
from tqdm.auto import tqdm

# uint16 covers vocab_size up to 65,536; fall back to uint32 automatically
# instead of hardcoding uint16 and hoping nobody raises vocab_size past it.
token_dtype = np.uint16 if tok.get_vocab_size() <= 65_536 else np.uint32

splits = {"train": train_docs, "val": val_docs}
CHUNK = 10_000
for name, docs_split in splits.items():
    path = f"{cfg.data_dir}/{name}.bin"
    n_tokens, n_content = 0, 0
    # Write each chunk straight to disk instead of collecting every chunk into
    # one Python list and concatenating at the end -- that pattern holds the
    # whole split in RAM twice (the list, then the concatenated array).
    # Streaming to the file is the shape a real, much larger pipeline takes.
    with open(path, "wb") as fh:
        for s in tqdm(range(0, len(docs_split), CHUNK), desc=name):
            for enc in tok.encode_batch(docs_split[s:s+CHUNK]):
                ids = np.array(enc.ids + [EOT], dtype=token_dtype)
                fh.write(ids.tobytes())
                n_tokens += ids.size
                n_content += len(enc.ids)      # content tokens, excluding EOT
    n_bytes = sum(len(d.encode("utf-8")) for d in docs_split)
    json.dump({"n_tokens": n_tokens,           # includes one EOT per document
               "n_content_tokens": n_content,  # use THIS for BPB
               "n_bytes": n_bytes,
               "dtype": token_dtype.__name__}, # Stage 7's Loader reads this back
              open(f"{cfg.data_dir}/{name}_meta.json", "w"))
    print(f"{name}: {n_tokens/1e6:.2f}M tokens ({token_dtype.__name__}) -> {path}")

meta = json.load(open(f"{cfg.data_dir}/train_meta.json"))
# "Equivalent" epochs, not literal ones -- Stage 7's loader samples random
# offsets WITH replacement, so this is token-exposures relative to corpus
# size, not a count of sequential passes through it.
equiv_epochs = TOKENS_PER_STEP * cfg.max_steps / meta["n_tokens"]
print(f"\ntraining exposure: {equiv_epochs:.2f} equivalent token epochs")

**Verify.** The first 200 tokens decode to readable text (using `token_dtype`,
not a hardcoded `np.uint16`), with `<|endoftext|>` at document boundaries.
Check `equiv_epochs`: at most ~4 — otherwise get more data (`n_docs`) or run
fewer steps.

**Breaks like this.** Hardcoding `np.uint16` for both writing and reading, so a
later vocab-size increase past 65,536 silently wraps token ids instead of
failing loudly -- this is why `token_dtype` is computed once here and threaded
through the meta file to Stage 7. Forgetting the separator token, so the model
learns to run documents together.

In [ ]:
arr = np.memmap(f"{cfg.data_dir}/train.bin", dtype=token_dtype, mode="r")
print(tok.decode(arr[:200].tolist(), skip_special_tokens=False))
print(f"\n{int((arr == EOT).sum()):,} separators for {len(train_docs):,} train docs")
if equiv_epochs > 4:
    print(f"WARNING: {equiv_epochs:.1f} equivalent epochs is past the ~4 where "
          f"repetition stops helping")

## Stage 7 — The dataloader

**Goal.** `get_batch(step)` returning `(x, y)` — deterministic and resumable.

**What's happening.** A batch is `B` random windows of length `T+1` into the token
array; inputs are all but the last token, targets are all but the first. Keying
the RNG on the global step makes the whole data order a pure function of
`(seed, step)`, so resuming a crashed run needs only the step number
(guide, Part 9.6).

In [ ]:
# Cell 7 — memmap dataloader
class Loader:
    def __init__(self, path, batch_size, seq_len, seed=0, dtype=np.uint16):
        # dtype must match what Stage 6 wrote this file with, or every token
        # id silently reads back wrong. Pass it through explicitly rather than
        # hardcoding it, since Stage 6 picks uint16 vs uint32 from vocab_size.
        self.data = np.memmap(path, dtype=dtype, mode="r")
        self.B, self.T, self.seed = batch_size, seq_len, seed
        self.n_start = len(self.data) - seq_len - 1
        assert self.n_start > 0, "corpus shorter than one sequence"

    def get_batch(self, step, device):
        # RNG keyed on step => data order is a pure function of (seed, step)
        g = np.random.default_rng((self.seed, step))
        ix = g.integers(0, self.n_start, size=self.B)
        x = np.stack([self.data[i:i+self.T]       for i in ix]).astype(np.int64)
        y = np.stack([self.data[i+1:i+1+self.T]   for i in ix]).astype(np.int64)
        x, y = torch.from_numpy(x), torch.from_numpy(y)
        if device == "cuda":
            return (x.pin_memory().to(device, non_blocking=True),
                    y.pin_memory().to(device, non_blocking=True))
        return x.to(device), y.to(device)

token_dtype  = getattr(np, meta["dtype"])   # `meta` is train_meta.json, loaded
                                             # at the end of Stage 6; reused here.
train_loader = Loader(f"{cfg.data_dir}/train.bin", cfg.batch_size, cfg.seq_len,
                      cfg.seed,     dtype=token_dtype)
val_loader   = Loader(f"{cfg.data_dir}/val.bin",   cfg.batch_size, cfg.seq_len,
                      cfg.seed + 1, dtype=token_dtype)

x, y = train_loader.get_batch(0, device)
print(x.shape, y.shape, x.dtype)
print("x:", tok.decode(x[0, :40].tolist()))
print("y:", tok.decode(y[0, :40].tolist()))   # must be x shifted by exactly one
assert torch.equal(x[0, 1:], y[0, :-1]), "off-by-one in the targets"

a, _ = train_loader.get_batch(5, device)
b, _ = train_loader.get_batch(5, device)
assert torch.equal(a, b), "loader is not deterministic -> not resumable"

**Verify.** Both asserts pass: `y` is `x` shifted by exactly one token, and the
same step always gives the same batch. The off-by-one is the single most common
silent bug in pretraining — it still trains, just to a permanently worse loss.

**Breaks like this.** Sampling with replacement means occasional repeats — fine at
scale, worth knowing. Windows cross document boundaries; real runs often add
document masking, which is a good extension exercise.

## Stage 8 — Build the model

**Goal.** A modern decoder-only transformer: RMSNorm, RoPE, GQA, SwiGLU, QK-norm.

**What's happening.** This is a representative modern stack, not the 2017 one
(guide, Part 5). Pre-norm keeps a clean residual path; RoPE encodes relative
position by rotation; GQA shares K/V heads to shrink the inference cache; SwiGLU
gates the MLP; QK-norm bounds attention logits for stability.

In [ ]:
# Cell 8 — the model
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.eps = eps; self.g = nn.Parameter(torch.ones(d))
    def forward(self, x):
        dt = x.dtype; x = x.float()
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x * self.g.float()).to(dt)

def build_rope(head_dim, max_len, theta, device):
    inv = 1.0 / (theta ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len, device=device).float(), inv)
    return torch.cos(freqs), torch.sin(freqs)          # each (max_len, hd/2)

def apply_rope(x, cos, sin):
    # x: (B, n_head, T, hd); rotate-half convention (pairs i with i+hd/2)
    T = x.size(-2)
    cos, sin = cos[:T][None, None], sin[:T][None, None]
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1*cos - x2*sin, x1*sin + x2*cos], -1).type_as(x)

class Attention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_head % c.n_kv_head == 0
        self.nh, self.nkv = c.n_head, c.n_kv_head
        self.hd = c.d_model // c.n_head
        self.rep = self.nh // self.nkv
        self.wq = nn.Linear(c.d_model, self.nh  * self.hd, bias=False)
        self.wk = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.wv = nn.Linear(c.d_model, self.nkv * self.hd, bias=False)
        self.wo = nn.Linear(self.nh * self.hd, c.d_model, bias=False)
        self.q_norm, self.k_norm = RMSNorm(self.hd), RMSNorm(self.hd)   # QK-norm

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.nh,  self.hd).transpose(1, 2)
        k = self.wk(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.wv(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        q, k = self.q_norm(q), self.k_norm(k)
        # RoPE before the repeat: same result either way (it acts per-head with
        # shared cos/sin), but this rotates nkv heads instead of nh of them.
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k = k.repeat_interleave(self.rep, dim=1)      # GQA: broadcast KV heads
        v = v.repeat_interleave(self.rep, dim=1)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # FlashAttention
        return self.wo(y.transpose(1, 2).reshape(B, T, C))

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        h = int(8 * c.d_model / 3)                      # SwiGLU param-matching
        h = c.ffn_multiple * math.ceil(h / c.ffn_multiple)
        self.w_gate = nn.Linear(c.d_model, h, bias=False)
        self.w_up   = nn.Linear(c.d_model, h, bias=False)
        self.w_down = nn.Linear(h, c.d_model, bias=False)
    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.attn = RMSNorm(c.d_model), Attention(c)
        self.n2, self.mlp  = RMSNorm(c.d_model), MLP(c)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.n1(x), cos, sin)       # pre-norm residual
        return x + self.mlp(self.n2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__(); self.cfg = c
        self.embed  = nn.Embedding(c.vocab_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm_f = RMSNorm(c.d_model)
        self.head   = nn.Linear(c.d_model, c.vocab_size, bias=False)
        if c.tie_embeddings:
            self.head.weight = self.embed.weight
        cos, sin = build_rope(c.d_model // c.n_head, c.seq_len, c.rope_theta, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        # scale residual-output projections by 1/sqrt(2*n_layer)  [Part 7.4]
        for n, p in self.named_parameters():
            if n.endswith(("wo.weight", "w_down.weight")):
                nn.init.normal_(p, std=0.02 / math.sqrt(2 * c.n_layer))

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if getattr(m, "bias", None) is not None: nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        x = self.embed(idx)
        for b in self.blocks:
            x = b(x, self.cos, self.sin)
        x = self.norm_f(x)
        logits = self.head(x)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)).float(),
                               targets.reshape(-1))     # CE in fp32
        return logits, loss

    def n_params(self, non_embedding=True):
        # With tied weights the embedding table IS the LM head, so non_embedding=True
        # drops the head too. That matches N in the 6N approximation; the head's
        # FLOPs are added back separately in Stage 11's fuller estimate.
        n = sum(p.numel() for p in self.parameters())
        if non_embedding: n -= self.embed.weight.numel()
        return n

model = GPT(cfg).to(device)
N = model.n_params()
print(f"total {sum(p.numel() for p in model.parameters())/1e6:.2f}M  |  "
      f"non-embedding {N/1e6:.2f}M")
print(f"Chinchilla-optimal tokens ≈ {20*N/1e6:.0f}M   "
      f"(this run: {TOKENS_PER_STEP*cfg.max_steps/1e6:.0f}M)")

**Verify.** The non-embedding count matches the preset's advertised size and your
Stage 1 guess. A forward pass returns `logits` of shape `(B, T, vocab_size)`.
Look at the Chinchilla line and know which side of it you're on.

**Breaks like this.** `d_model` not divisible by `n_head`. Forgetting the fp32 cast
in cross-entropy. Mixing the rotate-half and interleaved-pair RoPE conventions —
both are correct but not interchangeable, and mixing them trains to a worse loss
without ever erroring.

In [ ]:
with torch.no_grad():
    logits, _ = model(x)                  # x from Stage 7
assert logits.shape == (*x.shape, cfg.vocab_size), logits.shape
print("forward pass OK:", tuple(logits.shape))

## Stage 9 — Pre-flight sanity checks

**Goal.** Prove the model can learn *before* spending an hour finding out it can't.

**What's happening.** Three cheap tests catch the overwhelming majority of
implementation bugs: the initial loss is right, no information leaks from the
future, and the model can memorize a tiny batch. This stage has the best
time-saved-per-line ratio in the whole notebook.

**Predict before you run.** What initial loss do you expect? (Hint: a fresh model
is close to uniform over the vocabulary.)

*Notebook note:* test 3 overfits 4 sequences rather than the whole batch, at the
preset's learning rate, so it passes reliably in seconds on every preset.

In [ ]:
# Cell 9 — pre-flight checks
# 1. Initial loss must be ≈ ln(vocab_size): a fresh model is uniform.
x, y = train_loader.get_batch(0, device)
with torch.no_grad():
    _, loss0 = model(x, y)
expected = math.log(cfg.vocab_size)
print(f"init loss {loss0.item():.4f}  expected ≈ {expected:.4f}")
assert abs(loss0.item() - expected) < 0.7, "bad init or broken forward pass"

# 2. Causality: changing a FUTURE token must not change an EARLIER prediction.
xa = x[:1].clone(); xb = xa.clone(); xb[0, -1] = (xb[0, -1] + 1) % cfg.vocab_size
with torch.no_grad():
    la, _ = model(xa); lb, _ = model(xb)
drift = (la[0, :-1] - lb[0, :-1]).abs().max().item()
print(f"causality drift {drift:.2e}")
assert drift < 1e-4, "information is leaking from the future"

# 3. Overfit a tiny batch: a correct model can memorize a handful of sequences.
#    Memorize 4 sequences, not the whole batch, and test a RELATIVE drop: an
#    absolute target like "< 0.5" depends on model size and step count, so it
#    can fail a healthy tiny model -- and a failed assert here blocks the run.
probe = GPT(cfg).to(device)
opt = torch.optim.AdamW(probe.parameters(), lr=cfg.lr)
xo, yo = x[:4], y[:4]
l0 = None
for i in range(300):
    _, l = probe(xo, yo); opt.zero_grad(); l.backward(); opt.step()
    if l0 is None: l0 = l.item()
    if i % 50 == 0: print(f"  overfit step {i:3d}  loss {l.item():.4f}")
print(f"overfit loss {l0:.3f} -> {l.item():.3f}")
assert l.item() < 0.25 * l0, "cannot memorize 4 sequences -> real bug, do not proceed"
del probe, opt
if device == "cuda": torch.cuda.empty_cache()

**Verify.** All three asserts pass. Test 1 catches init and forward bugs, test 2
masking bugs, test 3 gradient-flow bugs. If test 3 fails, nothing downstream
matters — debug here.

**Breaks like this.** An init loss *well above* `ln V` means the logits are too
large at initialization (init std too big, or a missing final norm). Misaligned
labels cannot move the initial loss — that is what the Stage 7 assert is for.
Overfit stalling near `ln V` means no gradient is reaching the parameters.

## Stage 12a — Checkpoint helpers (runs here, before Stage 10)

This is the first half of Stage 12. It sits here because the training loop in
Stage 11 calls `save_ckpt` and `load_ckpt`, so they must already be defined. The
round-trip test (12b) runs after training.

A checkpoint must carry model, optimizer, config, *and the step number* — the step
is what reconstructs the data order in Stage 7. Each checkpoint stores the *next*
step to run, so a resume never replays a batch.

In [ ]:
# Cell 12a — checkpoint helpers
# Both functions use the global `scaler`, which Stage 11 creates before it first
# calls either one.
def save_ckpt(model, optimizer, step, cfg, hist, path):
    raw = getattr(model, "_orig_mod", model)        # unwrap torch.compile
    tmp = path + ".tmp"
    torch.save({"model": raw.state_dict(),
                "optimizer": optimizer.state_dict(),
                "step": step,              # NEXT step to run -> reconstructs data order
                "cfg": asdict(cfg),
                "hist": hist,
                "scaler": scaler.state_dict(),       # fp16 loss scale ({} if bf16)
                # Both RNGs: harmless today (no dropout; the loader is keyed on
                # step), needed for deterministic dropout after a resume. A fully
                # exact resume also needs the data position (here: `step`) and
                # deterministic kernels; multi-GPU runs need every rank's RNG too.
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": (torch.cuda.get_rng_state_all()
                             if torch.cuda.is_available() else None)}, tmp)
    os.replace(tmp, path)     # atomic: a crash mid-write can't corrupt the file
    print(f"saved {path} @ step {step}")

def load_ckpt(path, device):
    # weights_only=False unpickles arbitrary Python objects (needed for cfg/hist),
    # which can execute code. Only load checkpoints you created or trust.
    ck = torch.load(path, map_location=device, weights_only=False)
    c = Config(**ck["cfg"])
    m = GPT(c).to(device); m.load_state_dict(ck["model"])
    o = make_optimizer(m, c); o.load_state_dict(ck["optimizer"])
    if ck.get("scaler"):             # else fp16 restarts at 65536 and re-backs-off
        scaler.load_state_dict(ck["scaler"])
    torch.set_rng_state(ck["torch_rng"].cpu())
    if ck.get("cuda_rng") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all([s.cpu() for s in ck["cuda_rng"]])
    if c.compile and device == "cuda":
        m = torch.compile(m)
    return m, o, ck["step"], c, ck["hist"]

## Stage 10 — Optimizer and schedule

**Goal.** AdamW with correct parameter groups, plus a WSD learning-rate schedule.

**What's happening.** Weight decay belongs on matrices but not on norms and biases.
Whether it belongs on *embeddings* is genuinely unsettled — the common `dim >= 2`
rule decays them. Here they are excluded explicitly, so code and comment agree
(guide, Part 7.1). The WSD schedule warms up, holds flat, then decays over the
final stretch — which lets you stop at any point and makes the decay phase a
natural place to switch to higher-quality data (guide, Part 7.2).

**Predict before you run.** How many tensors land in the no-decay group? Count the
RMSNorm gains: four per block (`n1`, `n2`, `q_norm`, `k_norm`) plus the final
norm — `4·n_layer + 1`.

In [ ]:
# Cell 10 — optimizer + schedule
def make_optimizer(model, c):
    raw = getattr(model, "_orig_mod", model)
    # Three groups, stated explicitly rather than inferred from p.dim():
    #   matrices -> decay | norms & biases -> no decay | embeddings -> choose
    # NOTE: embed.weight is 2-D, so a bare `p.dim() >= 2` rule WOULD decay it.
    emb_ids = {id(raw.embed.weight)}
    decay, nodecay, embeds = [], [], []
    for n, p in raw.named_parameters():
        if not p.requires_grad:     continue
        if id(p) in emb_ids:        embeds.append(p)
        elif p.dim() >= 2:          decay.append(p)
        else:                       nodecay.append(p)
    tot = lambda ps: sum(p.numel() for p in ps)
    print(f"decay   : {len(decay):3d} tensors, {tot(decay)/1e6:6.2f}M")
    print(f"no-decay: {len(nodecay):3d} tensors, {tot(nodecay)/1e3:6.1f}K")
    print(f"embed   : {len(embeds):3d} tensors, {tot(embeds)/1e6:6.2f}M  (wd=0 here)")
    return torch.optim.AdamW(
        [{"params": decay,   "weight_decay": c.weight_decay},
         {"params": nodecay, "weight_decay": 0.0},
         {"params": embeds,  "weight_decay": 0.0}],
        lr=c.lr, betas=(c.beta1, c.beta2), eps=1e-8,
        fused=(device == "cuda"))

def lr_at(step, c):
    """Warmup -> Stable -> Decay. Clamped: never returns a negative LR."""
    step = min(step, c.max_steps)                     # <- clamp past the end
    warm  = max(int(c.warmup_frac * c.max_steps), 1)
    decay = max(int(c.decay_frac  * c.max_steps), 1)
    stable_end = c.max_steps - decay
    lo = c.lr * c.min_lr_frac
    if step < warm:        return c.lr * (step + 1) / warm
    if step < stable_end:  return c.lr
    prog = min((step - stable_end) / decay, 1.0)
    return lo + (c.lr - lo) * (1 - prog)              # linear decay tail

optimizer = make_optimizer(model, cfg)
if cfg.compile and device == "cuda":
    model = torch.compile(model)

assert lr_at(cfg.max_steps + 5_000, cfg) >= 0, "schedule goes negative past the end"

import matplotlib.pyplot as plt
plt.plot([lr_at(s, cfg) for s in range(int(cfg.max_steps * 1.2))])
plt.axvline(cfg.max_steps, ls=":", c="k")
plt.xlabel("step"); plt.ylabel("lr"); plt.title("WSD schedule"); plt.show()

**Verify.** The no-decay count matches your prediction, embeddings appear on their
own line rather than hidden in `decay`, and the plot shows three phases and stays
flat at `min_lr` past `max_steps`.

**Breaks like this.** A `p.dim() >= 2` rule under a comment claiming embeddings
are excluded — they are not, and with tied weights that silently decays the LM
head too. An unclamped schedule that goes negative if anything trains past
`max_steps`.

In [ ]:
assert len(optimizer.param_groups[1]["params"]) == 4 * cfg.n_layer + 1, \
    "no-decay group should hold exactly the RMSNorm gains"

## Stage 11 — The training loop

**Goal.** A trained model, plus the instrumentation that tells you whether the run
is healthy.

**What's happening.** The loop is: sample → forward → scaled backward ×
`grad_accum` → clip → step → log. What separates a real run from a toy one is
everything around it — grad norm, MFU, eval loss — because these let you diagnose
a run instead of guessing (guide, Parts 9.5 and 10). The loop checkpoints every
`ckpt_every` steps and **resumes automatically** if `last.pt` exists. It also
saves `stable_end.pt` just before the LR decay begins; Stage 14 branches from it.

MFU is printed two ways: the fuller estimate (adds the LM head and attention
FLOPs, which matter a lot at this size) and bare `6N`, which is what most papers
report.

**Predict before you run.** What loss at step 10, and at step 100? (Step ~0 is
near `ln V`; within a hundred steps a working model is at or below the unigram
entropy, roughly 5–6 nats for English.) Guess your MFU too — on a small model you
are launch-latency bound, and 15–25% is normal.

In [ ]:
# Cell 11 — training loop with instrumentation
autocast = (torch.autocast("cuda", dtype=amp_dtype) if device == "cuda"
            else torch.autocast("cpu", enabled=False))
# fp16 generally needs dynamic loss scaling to avoid gradient underflow; bf16
# normally does not. Enabled only on pre-Ampere GPUs.
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and not use_bf16))

# Dense bf16/fp16 peak, no sparsity. Add your card if it's missing.
# Matching is by substring, so ORDER MATTERS: specific names before the names
# they contain ("L40S" before "L4", "H100 PCIe" before "H100", "A100" before "A10").
PEAK = [("H100 PCIe", 756e12), ("H100", 989e12), ("A100", 312e12),
        ("L40S", 362e12), ("L40", 181e12), ("L4", 121e12), ("A10", 125e12),
        ("T4", 65e12), ("V100", 125e12), ("4090", 165e12), ("3090", 71e12)]
PEAK_FLOPS = next((v for k, v in PEAK if k in gpu_name), 100e12)
print(f"assuming {PEAK_FLOPS/1e12:.0f} TFLOP/s peak for '{gpu_name}'")

def flops_per_token(c, n_nonemb):
    """6N undercounts at this scale: the LM head is ~36% of N for `small`,
    and attention is not a parameter matmul at all. [Part 9.5]"""
    head = c.vocab_size * c.d_model                      # tied head still costs FLOPs
    attn = 12 * c.n_layer * c.d_model * c.seq_len        # QK^T and AV, fwd+bwd
    return 6 * (n_nonemb + head) + attn

FLOPS_PER_TOKEN = flops_per_token(cfg, N)   # fuller estimate: head + attention
FLOPS_6N        = 6 * N                      # bare 6N: comparable to most papers

@torch.no_grad()
def estimate_loss(net, loader, n_batches=20):
    was_training = net.training
    net.eval(); tot = 0.0
    for i in range(n_batches):
        # Fixed, deterministic batches -- NOT disjoint: offsets are sampled with
        # replacement, so windows can overlap. Fine for a noisy estimate.
        xb, yb = loader.get_batch(10**6 + i, device)
        with autocast: _, l = net(xb, yb)
        tot += l.item()
    net.train(was_training); return tot / n_batches

hist = {"step": [], "loss": [], "lr": [], "gnorm": [], "mfu": [],
        # val kept as a SEPARATE (step, loss) series, not slotted into the
        # per-log arrays above -- see the note at the log/eval block below.
        "val_step": [], "val_loss": []}
start_step = 0

# --- resume, if a checkpoint is already on disk ---
resume_path = f"{cfg.out_dir}/last.pt"
if os.path.exists(resume_path):
    model, optimizer, start_step, cfg, hist = load_ckpt(resume_path, device)  # Stage 12
    print(f"resumed from step {start_step}")

STABLE_END = cfg.max_steps - int(cfg.decay_frac * cfg.max_steps)
model.train(); t0 = time.time()
last_log = start_step - 1   # so the FIRST log's step count is inclusive of
                             # every step run since start_step, even if
                             # start_step isn't itself a multiple of 10 (an
                             # unusual config, but the arithmetic should hold)

for step in range(start_step, cfg.max_steps):
    lr = lr_at(step, cfg)
    for g in optimizer.param_groups: g["lr"] = lr

    loss_accum = torch.zeros((), device=device)         # stays on GPU: no sync
    for micro in range(cfg.grad_accum):
        xb, yb = train_loader.get_batch(step * cfg.grad_accum + micro, device)
        with autocast:
            _, loss = model(xb, yb)
        scaler.scale(loss / cfg.grad_accum).backward()
        loss_accum += loss.detach() / cfg.grad_accum    # .detach(), not .item()

    scaler.unscale_(optimizer)                          # unscale before clipping
    gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
    scaler.step(optimizer); scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if step % 10 == 0:                                  # the only per-step sync
        if device == "cuda": torch.cuda.synchronize()
        dt = time.time() - t0; t0 = time.time()
        n_steps = max(step - last_log, 1); last_log = step
        tok_s = TOKENS_PER_STEP * n_steps / dt
        mfu    = FLOPS_PER_TOKEN * tok_s / PEAK_FLOPS   # fuller estimate [Part 9.5]
        mfu_6n = FLOPS_6N        * tok_s / PEAK_FLOPS   # bare 6N, for comparisons
        l = loss_accum.item()
        hist["step"].append(step); hist["loss"].append(l)
        hist["lr"].append(lr);     hist["gnorm"].append(gnorm.item())
        hist["mfu"].append(mfu)
        print(f"step {step:5d} | loss {l:.4f} | lr {lr:.2e} | "
              f"gnorm {gnorm.item():5.2f} | {tok_s/1e3:6.1f}k tok/s | "
              f"mfu {mfu:5.1%} (6N {mfu_6n:5.1%})")

    # val is its OWN (step, loss) series -- not "the most recently logged
    # training-log entry" -- so this is correct however eval_every relates to
    # the training-log interval of 10 above.
    t_pause = time.time()      # eval and checkpoint I/O are NOT training work:
                               # excluded from the throughput timer at the end of the loop body
    if step % cfg.eval_every == 0 and step > start_step:
        v = estimate_loss(model, val_loader)
        hist["val_step"].append(step); hist["val_loss"].append(v)
        print(f"  >>> step {step}: val loss {v:.4f}  ppl {math.exp(v):.2f}")

    # Checkpoints store the NEXT step to run (step + 1). This step's update is
    # already applied; storing `step` would replay its batch on resume.

    # --- periodic checkpoint, so a crash costs at most ckpt_every steps ---
    if (step + 1) % cfg.ckpt_every == 0:
        save_ckpt(model, optimizer, step + 1, cfg, hist, resume_path)   # Stage 12

    # --- snapshot the end of the STABLE phase: Stage 14 branches from here ---
    if step + 1 == STABLE_END:      # last stable step done; next one starts decay
        save_ckpt(model, optimizer, step + 1, cfg, hist, f"{cfg.out_dir}/stable_end.pt")
        print(f"  >>> saved stable-phase checkpoint (decay starts at step {step + 1})")

    t0 += time.time() - t_pause    # shift the timer past eval/checkpoint time, so
                                   # tok/s and MFU show no artificial dips there

save_ckpt(model, optimizer, cfg.max_steps, cfg, hist, f"{cfg.out_dir}/final.pt")

**Verify.** Watch these four, in this order of importance:

| Signal | Healthy | Meaning if not |
|---|---|---|
| Loss at step ~100 | below the unigram entropy (~5–6 nats) | data or label bug |
| Grad norm | settles to a stable band, spikes rare | instability brewing (guide, Part 10) |
| MFU (fuller estimate) | `small`: 15–25%; `tiny` (0.8M params): only ~2–5% — measured 4% on a T4, launch-bound; 35–55% on a large, well-shaped run | dataloader stall, no compile, bad shapes |
| Val − train loss | small and stable | too few tokens per parameter |

**Breaks like this.** Calling `.item()` on every micro-step forces a GPU sync and
quietly costs throughput. Clipping *before* `scaler.unscale_()`, which clips the
scaled gradients against an arbitrary threshold. Hardcoding an H100's peak FLOPs
and reporting an MFU that is 15× too low on a T4. Forgetting `/ cfg.grad_accum`
does *not* inflate the effective LR under Adam — it inflates the gradient norm, so
clipping fires on most steps and you lose it as a diagnostic (guide, Part 7.3).
Storing val loss as `hist["val"][-1]` right after appending a training-log row --
correct only when `eval_every` is a multiple of the log interval, and silently
wrong the moment it isn't; keeping `val_step`/`val_loss` as their own series
avoids the coupling entirely.

## Stage 12b — Prove the checkpoint round trip

**Goal.** Confirm that `final.pt` reloads to the same loss *and* keeps the Adam
moments — saving weights without the optimizer causes a visible loss bump on
resume.

In [ ]:
# Cell 12b — prove the round trip actually works
m2, o2, s2, c2, h2 = load_ckpt(f"{cfg.out_dir}/final.pt", device)
xb, yb = val_loader.get_batch(0, device)
was_training = model.training
model.eval(); m2.eval()      # deterministic mode, so this stays valid if dropout is added
with torch.no_grad():
    _, l1 = model(xb, yb)
    _, l2 = m2(xb, yb)
model.train(was_training)
print(f"reloaded {l2.item():.6f} vs original {l1.item():.6f}  (step {s2})")
assert abs(l1.item() - l2.item()) < 1e-3, "checkpoint round-trip is broken"

# The optimizer state matters as much as the weights: confirm Adam's moments
# survived, or resuming will visibly bump the loss.
st = next(iter(o2.state.values()))
assert "exp_avg" in st and st["exp_avg"].abs().sum() > 0, "Adam moments lost"
print("optimizer moments restored OK")
del m2, o2

**Verify.** Both asserts pass. The real test is a restart: see the **Appendix** at
the bottom of the notebook.

**Breaks like this.** Saving `torch.compile`'s wrapper, which prefixes every key
with `_orig_mod.`. Saving the model but not the optimizer. Storing the step you
just *finished* instead of the next one, so every resume replays one batch.
Writing the checkpoint non-atomically, so a crash mid-save loses the run. Saving
only at the end — which is not checkpointing, it's exporting.

## Stage 13 — Evaluate

**Goal.** Held-out loss, bits-per-byte, and generated samples.

**What's happening.** Raw loss is only comparable between identical tokenizers;
BPB normalizes by the underlying bytes, which makes models with different
tokenizers comparable on the same text (guide, Part 6). Generation is the qualitative check — numbers can look fine while
output is degenerate.

**Predict before you run.** Write down your expected val loss and BPB, and what a
sample will read like.

In [ ]:
# Cell 13 — evaluation
val_meta = json.load(open(f"{cfg.data_dir}/val_meta.json"))

@torch.no_grad()
def evaluate_corpus(model, data, T, bs=4):
    """Score every target token once, in non-overlapping BLOCKS of length T.
    Context is reset at each block boundary: a token at position 700 is predicted
    from tokens 512..699, not from 0..699. That is the standard fixed-block
    convention; it slightly overestimates the loss compared with sliding-window
    evaluation (stride < T, scoring only the new targets), which is costlier.
    Returns (sum of NLL in nats over CONTENT targets, number of such targets).
    Targets equal to EOT are masked out: the injected separators have no bytes,
    so their loss must not enter a bits-per-BYTE numerator. (`estimate_loss`
    cannot do this -- it averages over every target in randomly sampled windows,
    EOT included, and may measure some tokens twice and others never.)"""
    net = getattr(model, "_orig_mod", model)        # uncompiled: window shapes vary
    was_training = net.training; net.eval()
    # `data` stays a memmap; only the window being scored is converted to int64.
    a64 = lambda lo, hi: torch.from_numpy(np.asarray(data[lo:hi], dtype=np.int64))
    n = len(data) - 1                                # number of (input, target) pairs
    nll, cnt = 0.0, 0

    def run(x, y):
        with autocast: logits, _ = net(x.to(device))
        y = y.to(device).reshape(-1)
        l = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y,
                            reduction="none")
        keep = y != EOT
        return l[keep].sum().item(), int(keep.sum())

    starts = list(range(0, n - T + 1, T))            # full windows
    for b in range(0, len(starts), bs):
        idx = starts[b:b + bs]
        a, c = run(torch.stack([a64(s, s + T)         for s in idx]),
                   torch.stack([a64(s + 1, s + T + 1) for s in idx]))
        nll += a; cnt += c
    tail = (starts[-1] + T) if starts else 0
    if tail < n:                                     # shorter remainder window
        a, c = run(a64(tail, n)[None], a64(tail + 1, n + 1)[None])
        nll += a; cnt += c
    net.train(was_training)
    return nll, cnt

nll, n_eval = evaluate_corpus(model, val_loader.data, cfg.seq_len)
# Every content token is scored except the file's very first (no context to
# predict it from), so n_eval should equal n_content_tokens - 1. Its bytes are
# still in n_bytes, so BPB differs from exact corpus BPB by one token's worth --
# negligible for any real validation set.
assert abs(n_eval - val_meta["n_content_tokens"]) <= 1, (n_eval, val_meta)
val_loss = nll / n_eval                              # nats per content token
bpb = nll / (math.log(2) * val_meta["n_bytes"])      # total content NLL / total bytes
print(f"val loss {val_loss:.4f} | ppl {math.exp(val_loss):.2f} | BPB {bpb:.4f}")

@torch.no_grad()
def generate(model, prompt, max_new=120, temp=0.8, top_k=50):
    # Notebook note: sample with the uncompiled module. The prompt grows by one
    # token per step, which would make torch.compile recompile repeatedly.
    net = getattr(model, "_orig_mod", model)
    was_training = net.training
    net.eval()
    ids = torch.tensor([tok.encode(prompt).ids], device=device)
    for _ in range(max_new):
        window = ids[:, -cfg.seq_len:]
        with autocast: logits, _ = net(window)
        logits = logits[:, -1, :].float() / temp
        if top_k:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")
        nxt = torch.multinomial(F.softmax(logits, -1), 1)
        if nxt.item() == EOT: break
        ids = torch.cat([ids, nxt], dim=1)
    net.train(was_training)                          # restore, don't force train mode
    return tok.decode(ids[0].tolist())

for p in ["Once upon a time", "The little robot"]:
    print(f"\n--- {p!r} ---\n{generate(model, p)}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].plot(hist["step"], hist["loss"], lw=.8, label="train")
if hist["val_step"]:
    ax[0].plot(hist["val_step"], hist["val_loss"], "o-", label="val")
ax[0].set_yscale("log"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(hist["step"], hist["gnorm"], lw=.8); ax[1].set_title("grad norm")
ax[2].plot(hist["step"], hist["mfu"],  lw=.8); ax[2].set_title("MFU (fuller estimate)")
for a in ax: a.set_xlabel("step")
plt.tight_layout(); plt.show()

**Verify.** On TinyStories at `small`, expect val loss ~1.3–1.8 and grammatical,
mostly coherent stories. At `tiny` the loss is much higher and the stories only
partly hold together — that is expected for a smoke test. Degenerate repetition
can indicate undertraining or too low a sampling temperature -- but also data
repetition, a missing/mishandled EOS token, or (less likely here) training
instability. Check the val loss and grad norm history above first.

**Breaks like this.** Comparing raw loss across runs with different vocab sizes —
always use BPB. Evaluating without `model.eval()` once you add dropout.
Computing BPB as `mean_loss × n_content_tokens / (ln 2 × n_bytes)` from a loss
averaged over windows *including* EOT targets: numerator and denominator then
count different tokens. Sum the content-only NLL and divide by bytes.

## Stage 14 — Anneal (a controlled A/B)

**Goal.** A *controlled* measurement of whether annealing on higher-quality data
beats simply finishing the run normally.

**What's happening.** The last 10–20% of training, at decaying LR on a
higher-quality mix, moves downstream quality disproportionately (guide, Part 13).
The experimental design is the whole lesson. Two traps make the naive version
worthless:

1. **You cannot anneal an already-annealed model.** Stage 11's schedule already
   decayed to `min_lr`. Decaying *again* measures "more steps at a low LR", not
   "better data". So both arms branch from the **same stable-phase checkpoint** —
   `stable_end.pt`.
2. **You need a control arm.** Run the identical decay on the *original* mix.
   Without it, any improvement can't be attributed to the data.

**Predict before you run.** Write down the delta you expect, with a sign. On
TinyStories — already clean and homogeneous — an honest prediction is "close to
zero, possibly negative". A near-zero result is a *correct* experiment, not a
failed one. So is a clearly negative one: a first `tiny` run on a T4 gave control
2.745 vs anneal 2.890 (−0.146 nats). One plausible reason -- not tested -- is that
the top-scoring 1.9k documents are a narrower slice than the full corpus.

In [ ]:
# Cell 14 — annealing, as a controlled A/B
# Quality proxy. NOTE: stopword *count* would just select the longest documents;
# the ratio is length-independent. On TinyStories this proxy is weak by design —
# the control arm is what makes the result interpretable.
def quality_score(d):
    s = quality_signals(d)
    return s["n_stopwords"] / max(s["n_words"], 1) + 0.5 * s["frac_terminal"]

DECAY_STEPS = int(cfg.decay_frac * cfg.max_steps)

# Size the anneal corpus to the decay phase. A fixed 20k docs (~4M tokens) against
# a ~59M-token decay would expose each token ~15 times over -- far past the ~4
# in [Part 3.5] -- and the arm would lose to plain repetition, not to bad data.
# (Sampling is random-with-replacement, so these are equivalent/expected
# exposures, not literal sequential epochs.)
MAX_ANNEAL_EPOCHS = 2
decay_tokens = DECAY_STEPS * TOKENS_PER_STEP
tok_per_doc  = meta["n_tokens"] / len(train_docs)
n_anneal = min(len(train_docs),
               math.ceil(decay_tokens / (MAX_ANNEAL_EPOCHS * tok_per_doc)))

# train_docs ONLY. Building this from `deduped` would include the validation
# split and contaminate every number below.
anneal_docs = sorted(train_docs, key=quality_score, reverse=True)[:n_anneal]
print(f"anneal corpus: top {n_anneal:,} of {len(train_docs):,} train docs "
      f"(~{decay_tokens / (n_anneal * tok_per_doc):.1f} equivalent epochs over the decay)")
# Compare CONTENT, not id(): an id() check is true by construction and can never
# fail. This one catches a val document that also appears in train by text.
assert not (set(anneal_docs) & set(val_docs)), "val leaked in"

np.concatenate([np.array(e.ids + [EOT], dtype=token_dtype)   # from Stage 6/7
                for e in tok.encode_batch(anneal_docs)]).tofile(
                f"{cfg.data_dir}/anneal.bin")
anneal_loader = Loader(f"{cfg.data_dir}/anneal.bin", cfg.batch_size, cfg.seq_len, 7,
                       dtype=token_dtype)

def decay_from_stable(loader, tag):
    """Branch from the end of the stable phase and run the SAME decay on `loader`."""
    m, o, step0, c, _ = load_ckpt(f"{cfg.out_dir}/stable_end.pt", device)
    m.train()
    for s in range(DECAY_STEPS):
        lr = lr_at(step0 + s, c)                  # the real WSD decay tail
        for g in o.param_groups: g["lr"] = lr
        for micro in range(c.grad_accum):
            xb, yb = loader.get_batch(10**7 + s * c.grad_accum + micro, device)
            with autocast: _, loss = m(xb, yb)
            scaler.scale(loss / c.grad_accum).backward()
        scaler.unscale_(o)
        torch.nn.utils.clip_grad_norm_(m.parameters(), c.grad_clip)
        scaler.step(o); scaler.update(); o.zero_grad(set_to_none=True)
        if s % 100 == 0: print(f"  [{tag}] {s:4d}/{DECAY_STEPS} lr {lr:.2e}")
    v = estimate_loss(m, val_loader, 50)
    print(f"  [{tag}] val loss {v:.4f}")
    return m, v

print("control arm: decay on the ORIGINAL mix")
m_ctrl,   v_ctrl   = decay_from_stable(train_loader,  "control")
print("treatment arm: decay on the HIGH-QUALITY mix")
m_anneal, v_anneal = decay_from_stable(anneal_loader, "anneal")

print(f"\ncontrol {v_ctrl:.4f}   anneal {v_anneal:.4f}   "
      f"delta {v_ctrl - v_anneal:+.4f} nats")
print("\n--- control ---\n",  generate(m_ctrl,   "Once upon a time"))
print("\n--- anneal  ---\n",  generate(m_anneal, "Once upon a time"))

**Verify.** The two arms differ only in data. If the delta is within run-to-run
noise, your proxy didn't separate quality — the common outcome on a clean corpus,
and the reason this technique is demonstrated on real web data. Re-run both arms
with a different seed to estimate that noise before believing any delta.

**Breaks like this.** Building the anneal corpus from `deduped` instead of
`train_docs`, which leaks validation documents into training. Annealing on top of
an already-decayed model. Ranking by a length-correlated statistic. Using an anneal
corpus so small it repeats many times over the decay. Reporting one arm with no
control.

## Stage 15 — Fit a scaling law

**Goal.** Predict the loss of a model you have not trained yet.

**What's happening.** This is the core professional skill in pretraining: you
de-risk expensive runs by forecasting them from cheap ones (guide, Part 8). Every
size trains to the *same tokens-per-parameter ratio*, so every point sits at
comparable convergence; the fit uses all but the largest, then predicts it.

**Budget.** At `small` this is ~1B tokens in total — roughly 3–4× Stage 11. At
`tiny` the notebook swaps in smaller sizes and a lower tokens-per-parameter ratio
so the stage finishes in minutes; treat that fit as a demonstration of the
mechanics, not a real forecast.

**Predict before you run.** Once the smaller points print, sketch them on a
log-log plot by eye and write down your prediction for the largest *before* the
fit prints its own.

In [ ]:
# Cell 15 — mini scaling law
from scipy.optimize import curve_fit

# Each model trains to the SAME tokens-per-parameter ratio, so every point is at
# comparable convergence. Training all sizes for a fixed number of steps would
# give them equal tokens, leaving the largest the most under-trained -- which
# bends the curve and is the usual reason a notebook scaling fit "fails".
TOK_PER_PARAM = 20                     # Chinchilla-ish [Part 8.2]
SIZES = [(128, 4), (192, 6), (256, 6), (320, 8), (384, 8), (512, 8)]
if PRESET == "tiny":
    # Notebook-only: smaller sizes and a lower ratio keep the smoke test to minutes.
    TOK_PER_PARAM = 5
    SIZES = [(64, 2), (96, 3), (128, 4), (160, 4), (192, 4)]

train_tokens = json.load(open(f"{cfg.data_dir}/train_meta.json"))["n_tokens"]
results = []
for d, L in SIZES:
    n_head = max(4, d // 64)
    # GQA needs n_head % n_kv_head == 0. d=320 gives 5 heads, which 2 KV heads
    # cannot divide -- fall back to a single KV head (MQA) for odd head counts.
    n_kv = 2 if n_head % 2 == 0 else 1
    c = Config(**{**PRESETS[PRESET], "d_model": d, "n_layer": L,
                  "n_head": n_head, "n_kv_head": n_kv})
    m = GPT(c).to(device); n = m.n_params()
    tps = c.batch_size * c.grad_accum * c.seq_len
    c.max_steps = max(200, round(TOK_PER_PARAM * n / tps))   # compute-matched
    equiv_epochs = c.max_steps * tps / train_tokens   # equivalent, not literal --
                                                       # see Stage 6's comment
    if equiv_epochs > 4:  # [Part 3.5] -- repeated data, not model size, will set this loss
        print(f"  WARNING: d={d} needs {equiv_epochs:.1f} equivalent epochs; "
              f"this point is data-limited and will sit ABOVE the true curve")
    o = make_optimizer(m, c)
    if c.compile and device == "cuda": m = torch.compile(m)
    m.train()
    for step in range(c.max_steps):
        for g in o.param_groups: g["lr"] = lr_at(step, c)
        for micro in range(c.grad_accum):
            xb, yb = train_loader.get_batch(step * c.grad_accum + micro, device)
            with autocast: _, loss = m(xb, yb)
            scaler.scale(loss / c.grad_accum).backward()
        scaler.unscale_(o)
        torch.nn.utils.clip_grad_norm_(m.parameters(), c.grad_clip)
        scaler.step(o); scaler.update(); o.zero_grad(set_to_none=True)
    v = estimate_loss(m, val_loader, 30)
    results.append((n, v))
    print(f"N={n/1e6:6.2f}M  steps={c.max_steps:5d}  "
          f"tokens={c.max_steps*tps/1e6:7.1f}M  eq.epochs={equiv_epochs:4.1f}  loss={v:.4f}")
    del m, o
    if device == "cuda": torch.cuda.empty_cache()

Ns = np.array([r[0] for r in results]); Ls = np.array([r[1] for r in results])
law = lambda N, E, A, alpha: E + A * N ** (-alpha)
# Fit on all but the largest, then predict it. With 4-5 points and 3 parameters
# this is thin -- treat a good hit as encouraging, not as validation.
(E, A, alpha), _ = curve_fit(law, Ns[:-1], Ls[:-1], p0=[1.0, 100.0, 0.3], maxfev=20000)
pred = law(Ns[-1], E, A, alpha)
print(f"\nL(N) = {E:.3f} + {A:.1f}·N^-{alpha:.3f}   (fit on {len(Ns)-1} points)")
print(f"held-out largest: predicted {pred:.4f}  actual {Ls[-1]:.4f}  "
      f"({100*abs(pred-Ls[-1])/Ls[-1]:.1f}% error)")

plt.loglog(Ns, Ls, "o", label="measured")
grid = np.logspace(np.log10(Ns[0]), np.log10(Ns[-1]*4), 50)
plt.loglog(grid, law(grid, E, A, alpha), "--", label="fit + extrapolation")
plt.loglog([Ns[-1]], [pred], "x", ms=12, label="prediction")
plt.xlabel("non-embedding params"); plt.ylabel("val loss"); plt.legend(); plt.show()

**Verify.** The held-out prediction lands within a few percent. The printed token
counts scale with `N` — if they are all equal, compute-matching broke. No size
should warn about equivalent epochs; if one does at `small`, raise `n_docs`
toward the full TinyStories train split (~2.1M stories) and re-run from Stage 2.

**Breaks like this.** Training every size for the same number of steps, so larger
models are under-trained and `alpha` comes out too small. Treating a 3-parameter
fit on a handful of points as confirmed. Letting the largest sizes run past ~4
equivalent epochs, so repetition bends the curve at exactly the point you are
predicting. Reusing one learning rate across all widths, which biases the
largest models (guide, Part 7.5).

## Stage 16 — Graduating from the notebook

You now have the whole pipeline. What a notebook structurally cannot teach:

| Next capability | Where it lives | Start with |
|---|---|---|
| Multi-GPU (FSDP/TP/PP) | a real cluster | `huggingface/nanotron`, `pytorch/torchtitan` |
| Trillion-token data | distributed processing | `huggingface/datatrove` |
| Fault tolerance | long runs on flaky hardware | `allenai/OLMo-core` |
| Kernel-level speed | Triton / CUDA | `linkedin/Liger-Kernel`, `karpathy/llm.c` |
| Current architecture tricks | the speedrun | `KellerJordan/modded-nanogpt` |

The natural port: take Stages 7–12, convert them to a script with `torchrun` +
FSDP2, swap TinyStories for a FineWeb slice, and run the `base` preset for a day.

---

## Appendix — Resume after a disconnect (Stage 12c)

Colab sessions drop. If you set `USE_DRIVE = True`, your tokenizer, token files and
checkpoints are still on Drive. **Do not re-run Stages 2–6** — they take a while
and their only outputs training needs are already saved. Instead:

1. Run **Stage 0** and **Stage 1** (same `PRESET`).
2. Run the cell below.
3. Run **Stage 7**, **Stage 8**, **Stage 12a**, **Stage 10**, then **Stage 11**.
   It prints `resumed from step N` and continues from that loss instead of
   starting over. Delete `last.pt` to start fresh.

Stages 13 and 15 work after this path too. Stage 14 needs `train_docs`,
`val_docs` and `quality_signals`, which only exist in memory, so it needs Stages
2–6 in the same session — they are deterministic, so re-running them reproduces
the same token files.

In [ ]:
# Cell 12c — fast resume after a kernel restart (replaces cells 2-6)
from tokenizers import Tokenizer
need = ["tokenizer.json", "train.bin", "val.bin", "train_meta.json", "val_meta.json"]
missing = [f for f in need if not os.path.exists(f"{cfg.data_dir}/{f}")]
assert not missing, f"missing {missing}: run Stages 2-6 once first"
tok  = Tokenizer.from_file(f"{cfg.data_dir}/tokenizer.json")
EOT  = tok.token_to_id("<|endoftext|>")
meta = json.load(open(f"{cfg.data_dir}/train_meta.json"))
print(f"loaded tokenizer + token files; last.pt present: "
      f"{os.path.exists(f'{cfg.out_dir}/last.pt')}")